# 경제 분석 및 예측과 데이터 지능 실습3 (sktime 버전): VAR Basics

기존 실습의 `statsmodels VAR` 사용 부분을 **sktime의 `VAR` forecaster**로 재구성합니다.

이 버전은 외부 API 다운로더(`oecddatabuilder`)를 제거하고, 레포에 포함된 CSV 데이터로만 진행합니다.

사용 데이터:
- `../datasets/querterly_oecd_account_data.csv`
- `../datasets/df_blanchard.csv`

핵심:
- 다변량 시계열을 VAR로 적합
- lag 선택을 `maxlags` + `ic`로 제어
- 예측 결과 시각화

In [ ]:
# --- Import hygiene (repo contains ./sktime source checkout) ---
import os
import sys

repo_root = os.path.abspath(os.getcwd())
local_sktime_src = os.path.join(repo_root, "sktime")
if os.path.exists(os.path.join(local_sktime_src, "sktime", "__init__.py")):
    sys.path = [p for p in sys.path if p not in ("", repo_root)]
    sys.path.insert(0, local_sktime_src)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sktime.forecasting.var import VAR
from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.model_selection import temporal_train_test_split

plt.rcParams["figure.figsize"] = (10, 4)

## 1) 데이터 불러오기 및 전처리 (Quarterly)

VAR 입력은 `(시간 index) x (변수 columns)` 형태의 DataFrame 입니다.
예시로 한 국가(USA)의 `Y, C`를 사용합니다.

In [ ]:
df = pd.read_csv("../datasets/querterly_oecd_account_data.csv", parse_dates=["date"])
df["date"] = df["date"].dt.to_period("Q")

country = "USA"
vars_to_use = ["Y", "C"]

wide = (
    df[df["country"] == country]
    .set_index("date")[vars_to_use]
    .sort_index()
)
wide = wide.asfreq("Q").interpolate(limit_direction="both")
wide.head()

In [ ]:
wide.plot(subplots=True, title=[f"{country} {c}" for c in wide.columns])
plt.tight_layout()
plt.show()

## 2) 로그 변환 + 1차 차분

비정상성을 완화하기 위해 로그 차분을 사용합니다.

In [ ]:
wide_log_diff = np.log(wide).diff().dropna()
wide_log_diff.head()

## 3) VAR 적합 (고정 lag)

sktime `VAR`는 `ic`로 자동 lag 선택도 가능하지만, 데이터/샘플 길이에 따라 **0 lag**가 선택되어 예측이 실패하는 케이스가 있습니다.

여기서는 실습의 안정성을 위해 **고정 lag**를 사용합니다.

- **lag(시차)**: `maxlags=4` (VAR(4))

In [ ]:
y_train, y_test = temporal_train_test_split(wide_log_diff, test_size=12)
fh = ForecastingHorizon(y_test.index, is_relative=False)

# Fixed lag VAR(4) for robust execution
var = VAR(maxlags=4, ic=None, trend="c")
var.fit(y_train)

y_pred = var.predict(fh)
y_pred.head()

In [ ]:
fig, axes = plt.subplots(nrows=len(wide_log_diff.columns), ncols=1, sharex=True)
if len(wide_log_diff.columns) == 1:
    axes = [axes]
for i, col in enumerate(wide_log_diff.columns):
    ax = axes[i]
    y_train[col].iloc[-40:].plot(ax=ax, label="train")
    y_test[col].plot(ax=ax, label="test")
    y_pred[col].plot(ax=ax, label="pred")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

## 4) (옵션) Blanchard-Quah 예시 데이터 로드

구조적 해석(Blanchard-Quah)은 여기서 완전히 구현하진 않고,
데이터를 VAR 입력 형태로 만드는 흐름만 보여줍니다.

In [ ]:
bq = pd.read_csv("../datasets/df_blanchard.csv", parse_dates=["date"])
bq["date"] = bq["date"].dt.to_period("Q")
bq = bq.set_index("date").sort_index()
bq.columns.tolist()